In [1]:
%cd ..

/home/gta/airam/MinicursoAprendizadoFederadoVeicularSBRC2026


/home/gta/miniconda3/envs/afv_sbrc_2026/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import flwr as fl
import torch

from collections import OrderedDict

from architectures.torch.implementation import train, evaluate
from utils.utils import load_config
from architectures.torch.implementation import build_model
from utils.torch.load_federated_data import load_data_client
from utils.torch.utils import create_logger_client

In [3]:
cfg = load_config("configs/config.yaml")

In [4]:
client_id_1 = 1
i_epochs = cfg['simulation']['federated_learning']['client']['local_epochs']
bs = 32
ts = 0.2
SERVER_IP = cfg['simulation']['federated_learning']['server']['ip']
SERVER_PORT = cfg['simulation']['federated_learning']['server']['port']
DATASET = cfg['simulation']['data']['name']
LOG_PATH = 'logs/clients/flwr/'
IMAGE_DATA = 1
MODEL = cfg['simulation']['model']['name']
num_clients = cfg['simulation']['cars']
num_selected_clients = cfg['simulation']['federated_learning']['server']['n_clients_selected']
alpha = cfg['simulation']['data']['alpha']
strategy = cfg['simulation']['federated_learning']['server']['strategy']


logger = create_logger_client(LOG_PATH+MODEL+'/', 
                              client_id_1)

message_length = 800 * 1024 * 1024

logger.debug("Loading dataset")
train_dataset_1, test_dataset_1 = load_data_client(dataset_name=DATASET, 
                                               clientID=client_id_1, 
                                               numClients=num_clients, 
                                               alpha=alpha,
                                               trPer=ts,
                                               distribution="dirichlet") 

trainloader_1 = torch.utils.data.DataLoader(train_dataset_1, 
                                            batch_size=bs, 
                                            shuffle=True,
                                            num_workers=0,
                                            pin_memory=True)

testloader_1 = torch.utils.data.DataLoader(test_dataset_1, 
                                           batch_size=bs, 
                                           shuffle=True,
                                           num_workers=0,
                                           pin_memory=True)

logger.debug("Building model")

labels = cfg['simulation']['data']['n_classes']
features_shape = int(cfg['simulation']['data']['shape'][-1])

model_1, criterion_1, optimizer_1, device_1, scheduler_1 = build_model(features_shape=features_shape,
                                                                       labels_shape=labels,
                                                                       model_name=MODEL,
                                                                       lr=0.1)    

In [5]:
class FLClient(fl.client.NumPyClient):

    def __init__(self, 
                 *args,
                 cid=-1,
                 model=None,
                 i_epochs=5,
                 model_name="MOBILENET",
                 batch_size=32,
                 dataset="CIFAR-10",
                 strategy="fedavg",
                 model_path="",
                 result_path="",
                 computation_time_path="",
                 logger=None,
                 optimizer=None,
                 criterion=None,
                 scheduler=None,
                 device=None,
                 trainloader=None,
                 testloader=None,
                 **kwargs):
        
        # paths
        self.strategy = strategy
        self.dataset = dataset
        self.model_name = model_name
        self.model_path = model_path+model_name+'/'
        self.result_path = result_path+model_name+'/'
        self.time_path = computation_time_path+model_name+'/'
        self.logger = logger
        self.global_epoch = 0
        
        # identifiers
        self.cid = cid

        self.model = model.to(device)
        self.optimizer = optimizer
        self.criterion = criterion
        self.scheduler = scheduler
        self.device = device

        # client's data
        self.trainloader = trainloader 
        self.testloader = testloader 
        self.train_size = len(trainloader.dataset)
        self.test_size = len(testloader.dataset)

        # learning parameters
        self.i_epochs = i_epochs
        self.bs = batch_size

    def get_weights(self):
        
        result = [val.cpu().numpy() for _, val in self.model.state_dict().items()]

        return result
        
    def set_weights(self, 
                    parameters):
    
        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.model.load_state_dict(state_dict, strict=True)

    # Implementar a função para o download de modelos
    def download_model(self):

        return 0

    # Implementar a função para o upload de modelos
    def upload_model(self):

        return 0

    # Implementar a função para o cálculo do atraso computacional
    def computational_time(self):

        return 0
    
    def get_properties(self, 
                       config):

        return {'cid': self.cid}

    def fit(self, 
            parameters, 
            config):

        communication_time = 0
        
        self.logger.debug("determine client's communication time to upload the model")
        communication_time += self.download_model()
        
        self.logger.debug("updating model parameters")
        self.set_weights(parameters)

        self.logger.debug("training model")
        loss = train(self.model, 
                     self.i_epochs, 
                     self.optimizer, 
                     self.criterion,
                     self.scheduler,
                     self.device,
                     self.trainloader,
                     self.logger)

        self.logger.debug("determine client's computational time")
        computational_time = computational_time()

        self.logger.debug("determine client's communication time to upload the model")
        communication_time += self.upload_model()

        total_training_time = computational_time + communication_time
        
        self.logger.debug(f'sending parameters to server: model_weights, len(train): {self.train_size}')
        return self.get_weights(), len(self.trainloader.dataset), {'loss':loss, "cid":self.cid, "training_time":total_training_time}

    def evaluate(self, 
                 parameters, 
                 config):
        
        self.logger.debug(f'evaluating model')  
        
        self.logger.debug("updating model parameters")
        self.set_weights(parameters)
 
        self.logger.debug("evaluating model")
        accuracy, loss = evaluate(self.model,
                                  self.device,
                                  self.criterion,
                                  self.testloader,
                                  self.logger)

        self.logger.debug(f'sending parameters to server: loss {loss}, len(test): {self.test_size} accuracy: {float(accuracy)}')
        return loss, self.test_size, {"accuracy": float(accuracy), "cid":self.cid}



In [6]:
fl.client.start_client(server_address=f'{SERVER_IP}:{SERVER_PORT}', 
                       client=FLClient(cid=client_id_1,                                    
                                       model=model_1,
                                       i_epochs=i_epochs,
                                       model_name=MODEL,
                                       batch_size=bs,
                                       dataset=DATASET,
                                       strategy=strategy,
                                       logger=logger,
                                       optimizer=optimizer_1,
                                       criterion=criterion_1,
                                       scheduler=scheduler_1,
                                       trainloader=trainloader_1,
                                       testloader=testloader_1,
                                       device=device_1).to_client(),
                                       grpc_max_message_length=message_length)


	Instead, use the `flower-supernode` CLI command to start a SuperNode as shown below:

		$ flower-supernode --insecure --superlink='<IP>:<PORT>'

	To view all available options, run:

		$ flower-supernode --help

	Using `start_client()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      
INFO :      Received: train message 0d79d3fc-822b-4397-a373-9f28ec305afb
ERROR :     Client raised an exception.
Traceback (most recent call last):
  File "/home/gta/miniconda3/envs/afv_sbrc_2026/lib/python3.12/site-packages/flwr/client/app.py", line 571, in start_client_internal
    reply_message = client_app(message=message, context=context)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/gta/miniconda3/envs/afv_sbrc_2026/lib/python3.12/site-packages/flwr/client/client_app.py", line 144, in __call__
    return self._call(message, context)
           ^^^^^^^^^^^^^^^^^^^^^^^^

TypeError: train() takes 7 positional arguments but 8 were given